In [2]:
%pip install matplotlib statsmodels scipy pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# 1. Cargar la base de datos
df = pd.read_csv("abandono_producto_financiero.csv")

# 2. Configurar los factores categóricos con sus niveles y referencias fijas
# Referencia de pais: Francia | Niveles: Francia, Alemania, España
df["pais"] = pd.Categorical(df["pais"], categories=["Francia", "Alemania", "España"])

# Referencia de sexo: Mujer | Niveles: Mujer, Hombre
df["sexo"] = pd.Categorical(df["sexo"], categories=["Mujer", "Hombre"])

# 3. Inspeccionar las primeras filas y resumen
print("Forma de la base de datos (filas, columnas):", df.shape)
print("\nTipos de datos de las variables:")
print(df.dtypes)
print("\nPrimeras 5 filas:")
df.head()


Note: you may need to restart the kernel to use updated packages.
Forma de la base de datos (filas, columnas): (10000, 14)

Tipos de datos de las variables:
numero_fila              int64
id_cliente               int64
apellido                   str
puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

Primeras 5 filas:


,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0


Resumen Exploratorio de la Base de Datos: Abandono de Productos Financieros

El conjunto de datos contiene información socio-demográfica y financiera de los clientes de una entidad bancaria, orientada a analizar la fuga o abandono (churn) de productos financieros.

1. Observaciones Iniciales (Muestra de Primeras 5 Filas)

Diversidad Geográfica y Demográfica: La muestra inicial evidencia clientes residentes en países como Francia y España, con edades maduras (rango entre 39 y 43 años en los registros observados).

Saldos nulos: Se observa presencia de registros con saldo igual a 0.00 (p. ej., clientes con ID 15634602 y 15701354), lo que sugiere la necesidad de tratar o analizar por separado a los clientes con cuentas inactivas o con saldo en cero.

Variable Objetivo (abandono): En las primeras 5 observaciones se registran 2 casos de abandono (abandono = 1) y 3 permanencias (abandono = 0), lo cual servirá de base para calibrar los modelos predictivos posteriores.

In [3]:
# Frecuencias y proporciones de la variable respuesta (abandono)
tabla_abandono = pd.crosstab(df["abandono"], columns="conteo")
tabla_abandono["proporcion"] = df["abandono"].value_counts(normalize=True)
print("Distribución de la variable respuesta (abandono):")
print(tabla_abandono)

# Resumen estadístico de variables continuas
print("\nResumen descriptivo de variables cuantitativas:")
print(df[["puntaje_crediticio", "edad", "antiguedad", "saldo", "numero_productos", "salario_estimado"]].describe())

Distribución de la variable respuesta (abandono):
col_0     conteo  proporcion
abandono                    
0           7963      0.7963
1           2037      0.2037

Resumen descriptivo de variables cuantitativas:
       puntaje_crediticio          edad    antiguedad          saldo  \
count        10000.000000  10000.000000  10000.000000   10000.000000   
mean           650.528800     38.921800      5.012800   76485.889288   
std             96.653299     10.487806      2.892174   62397.405202   
min            350.000000     18.000000      0.000000       0.000000   
25%            584.000000     32.000000      3.000000       0.000000   
50%            652.000000     37.000000      5.000000   97198.540000   
75%            718.000000     44.000000      7.000000  127644.240000   
max            850.000000     92.000000     10.000000  250898.090000   

       numero_productos  salario_estimado  
count      10000.000000      10000.000000  
mean           1.530200     100090.239881  
std 

## Sección 2: Análisis Univariado

### 1. Variable Respuesta (`abandono`)
* **Distribución:** La cartera muestra que el **79.63%** (7,963 clientes) permanece en el banco (`abandono = 0`), mientras que el **20.37%** (2,037 clientes) cerró su relación financiera (`abandono = 1`).
* **Implicación Metodológica:** Existe un desbalance moderado de clases (~80/20). Evaluar únicamente la exactitud (*accuracy*) resultaría engañoso, ya que un modelo ingenuo que prediga que nadie se va tendría ~80% de aciertos pero 0% de detección de fuga. Por ello, será necesario evaluar la **sensibilidad** y calibrar el umbral de probabilidad (p. ej. 0.4 o 0.5) para gestionar eficazmente la retención.

### 2. Variables Cuantitativas

* **Edad (`edad`):** La edad media de la cartera es de **38.9 años**, con un 50% central de los clientes ubicados entre los **32 y 44 años** (IQR). El rango abarca desde los 18 hasta los 92 años.
* **Saldo en Cuenta (`saldo`):** El promedio se ubica en **$76,485.89**. Destaca que al menos un **25% de los clientes registra un saldo de $0.00**, mientras que el 50% con mayor saldo alcanza hasta los **$250,898.09**.
* **Puntaje Crediticio (`puntaje_crediticio`):** Presenta una media de **650.5 puntos** (rango entre 350 y 850), reflejando una distribución centrada y estable en la calidad crediticia.
* **Antigüedad (`antiguedad`):** La relación promedio con la entidad es de **5.01 años**, distribuida de forma homogénea en el rango de 0 a 10 años.
* **Número de Productos (`numero_productos`):** Concentración alta en pocos productos; la mediana es **1 producto** y el 75% de la muestra mantiene máximo 2 productos (rango de 1 a 4).
* **Salario Estimado (`salario_estimado`):** Presenta un comportamiento cercano a una distribución uniforme entre **$11.58 y $199,992.48**, con un promedio de **$100,090.24**.

In [4]:
# 1. Tablas cruzadas y prueba Ji-cuadrado para variables categóricas
for var in ["pais", "sexo", "tiene_tarjeta", "miembro_activo"]:
    tabla = pd.crosstab(df[var], df["abandono"], normalize="index")
    chi2, p, dof, ex = stats.chi2_contingency(pd.crosstab(df[var], df["abandono"]))
    print(f"--- Relación {var} vs Abandono (p-valor Ji2: {p:.4f}) ---")
    print(tabla)
    print("\n")

# 2. Promedios por grupo de abandono para variables cuantitativas
print("Promedios de variables cuantitativas según Abandono (0 vs 1):")
print(df.groupby("abandono")[["edad", "saldo", "puntaje_crediticio", "salario_estimado", "numero_productos", "antiguedad"]].mean())

--- Relación pais vs Abandono (p-valor Ji2: 0.0000) ---
abandono         0         1
pais                        
Francia   0.838452  0.161548
Alemania  0.675568  0.324432
España    0.833266  0.166734


--- Relación sexo vs Abandono (p-valor Ji2: 0.0000) ---
abandono         0         1
sexo                        
Mujer     0.749285  0.250715
Hombre    0.835441  0.164559


--- Relación tiene_tarjeta vs Abandono (p-valor Ji2: 0.4924) ---
abandono              0         1
tiene_tarjeta                    
0              0.791851  0.208149
1              0.798157  0.201843


--- Relación miembro_activo vs Abandono (p-valor Ji2: 0.0000) ---
abandono               0         1
miembro_activo                    
0               0.731491  0.268509
1               0.857309  0.142691


Promedios de variables cuantitativas según Abandono (0 vs 1):
               edad         saldo  puntaje_crediticio  salario_estimado  \
abandono                                                                  


## Sección 3: Análisis Bivariado (Variables Categóricas vs. Abandono)

### 1. País de Residencia (`pais`)
* **Prueba Ji-cuadrado:** p-valor = $0.0000$ (Diferencia altamente significativa)[cite: 12].
* **Hallazgo:** **Alemania** presenta la tasa de abandono más alta con un **32.44%**, duplicando las tasas de **Francia** (**16.15%**) y **España** (**16.67%**)[cite: 12]. El país de residencia es un fuerte factor discriminante del riesgo.

### 2. Género (`sexo`)
* **Prueba Ji-cuadrado:** p-valor = $0.0000$ (Diferencia altamente significativa)[cite: 12].
* **Hallazgo:** Las **mujeres** registran una mayor tasa de abandono (**25.07%**) en comparación con los **hombres** (**16.46%**)[cite: 12].

### 3. Tenencia de Tarjeta de Crédito (`tiene_tarjeta`)
* **Prueba Ji-cuadrado:** p-valor = $0.4924$ (No es estadísticamente significativo al 5%)[cite: 12].
* **Hallazgo:** La proporción de abandono entre quienes tienen tarjeta (**20.18%**) y quienes no (**20.81%**) es prácticamente idéntica[cite: 12]. Esta variable no aporta poder explicativo individual al modelo.

### 4. Estado de Actividad (`miembro_activo`)
* **Prueba Ji-cuadrado:** p-valor = $0.0000$ (Diferencia altamente significativa)[cite: 13].
* **Hallazgo:** Existe una fuerte asociación entre la inactividad del cliente y la probabilidad de cierre de cuenta[cite: 13].

### 5. Comparativa Promedios (`numero_productos` y `antiguedad`)
* **Número de Productos:** Los clientes que permanecen tienen en promedio **1.54 productos**, mientras que los que abandonan tienen **1.48 productos**[cite: 13].
* **Antigüedad:** La antigüedad promedio es muy parecida entre ambos grupos (**5.03 años** para quienes permanecen vs. **4.93 años** para quienes se van)[cite: 13].

In [5]:
# Modelo Lineal de Probabilidad (OLS)
m_ols = smf.ols("abandono ~ edad", data=df).fit()

# Modelo Logístico Simple (Logit)
m_logit_simple = smf.logit("abandono ~ edad", data=df).fit()

print("--- Modelo OLS ---")
print(m_ols.summary().tables[1])

print("\n--- Modelo Logit Simple ---")
print(m_logit_simple.summary().tables[1])

Optimization terminated successfully.
         Current function value: 0.467553
         Iterations 6
--- Modelo OLS ---
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.2228      0.015    -15.014      0.000      -0.252      -0.194
edad           0.0110      0.000     29.767      0.000       0.010       0.012

--- Modelo Logit Simple ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.9286      0.103    -38.096      0.000      -4.131      -3.726
edad           0.0629      0.002     26.614      0.000       0.058       0.068


## Sección 4: Comparación entre el Modelo Lineal (OLS) y el Logit Simple (Edad)

Para analizar el impacto directo de la edad en la probabilidad de abandono, probé primero un modelo de probabilidad lineal por OLS y luego un modelo Logit simple.

### 1. Modelo de Probabilidad Lineal (OLS)
* **Resultado del modelo:** 
  Probabilidad de abandono = -0.2228 + 0.0110 * edad
* **Interpretación:** La edad resulta ser una variable muy significativa. El resultado indica que por cada año adicional de edad del cliente, su probabilidad de abandonar el banco aumenta en 1.10 puntos porcentuales.
* **Problema del OLS:** Aquí es claro por qué el modelo lineal no es el más adecuado: da un valor inicial de -0.2228. Si calculáramos la probabilidad para un cliente joven de 18 años, el resultado daría negativo (-2.48%), lo cual no tiene sentido en términos de probabilidad y muestra por qué necesitamos el modelo Logit.

---

### 2. Modelo Logit Simple
* **Resultado del modelo:**
  Log-Odds de abandono = -3.9286 + 0.0629 * edad
* **Interpretación:** El resultado para la edad es positivo (0.0629) y muy significativo, lo que confirma que a mayor edad existe un mayor riesgo de que el cliente se vaya del banco.
* **Lectura en Odds Ratio (OR):**
  Al calcular la razón de probabilidades (exp(0.0629) = 1.0649), se observa que por cada año extra de edad, la posibilidad relativa de abandonar el banco aumenta un 6.49%. Esto confirma que los clientes de más edad representan un mayor riesgo de retiro en la cartera.

In [6]:
# Regresión logística múltiple completa
formula = "abandono ~ puntaje_crediticio + C(pais) + C(sexo) + edad + antiguedad + saldo + numero_productos + tiene_tarjeta + miembro_activo + salario_estimado"
m_full = smf.logit(formula, data=df).fit()

print(m_full.summary())

# Odds Ratios e Intervalos de Confianza al 95%
params = m_full.params
conf = m_full.conf_int()
conf['OR'] = params
conf.columns = ['2.5%', '97.5%', 'OR']
odds_ratios = np.exp(conf[['OR', '2.5%', '97.5%']])

print("\nOdds Ratios (OR) e Intervalos de Confianza:")
print(odds_ratios)

Optimization terminated successfully.
         Current function value: 0.428068
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               abandono   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9988
Method:                           MLE   Df Model:                           11
Date:                Thu, 24 Sep 2026   Pseudo R-squ.:                  0.1532
Time:                        16:01:47   Log-Likelihood:                -4280.7
converged:                       True   LL-Null:                       -5054.9
Covariance Type:            nonrobust   LLR p-value:                     0.000
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -3.3923      0.245    -13.857      0.000      -3.872      -2.912
C(

# Sección 5: Modelo Logit Múltiple y Ratios de Probabilidad (Odds Ratios)

Estimé un modelo Logit múltiple incluyendo todas las variables explicativas para evaluar el riesgo de abandono manteniendo las demás variables constantes (ceteris paribus).

### 1. Ajuste General del Modelo
* **Ajuste:** El Pseudo R-cuadrado es de 0.1532 y la prueba global (LLR p-valor) es de 0.000, lo que confirma que el conjunto de variables explicativas aporta información relevante para predecir el abandono.

---

### 2. Significancia de las Variables
* **Variables Significativas (p < 0.05):**
  * **Alemania:** Presenta un coeficiente positivo (0.7747) y un p-valor de 0.000, lo que significa que vivir en Alemania incrementa significativamente el riesgo de abandono en comparación con Francia (país de referencia).
  * **Sexo (Hombre):** Coeficiente negativo (-0.5285, p = 0.000), indicando que los hombres tienen un riesgo menor de abandonar en comparación con las mujeres.
  * **Edad:** Coeficiente positivo (0.0727, p = 0.000), reconfirmando que a mayor edad aumenta el riesgo de retiro.
  * **Saldo:** Coeficiente positivo (2.637e-06, p = 0.000), mostrando un impacto positivo pero moderado sobre la probabilidad de abandono.
  * **Puntaje Crediticio y Número de Productos:** Ambos muestran coeficientes negativos (-0.0007 y -0.1015 respectivamente), indicando que un mejor puntaje o tener más productos ayuda a reducir el riesgo de abandono.

* **Variables No Significativas (p > 0.05):**
  * **España:** Su p-valor es de 0.618, por lo que no muestra diferencias estadísticamente significativas en el riesgo de abandono frente a Francia.
  * **Tiene Tarjeta:** Su p-valor es de 0.452, confirmando que poseer tarjeta de crédito no influye en la decisión de abandonar.
  * **Antigüedad:** Su p-valor es de 0.088, por lo que no es significativa al nivel estándar del 5%.

---

### 3. Interpretación de Odds Ratios Principales (Riesgo vs. Protección)
* **Factores de Riesgo (Odds Ratio > 1):**
  * **Alemania:** Tiene un OR mayor a 1, lo que confirma a la residencia en Alemania como uno de los principales factores que aumentan la posibilidad relativa de abandono.
  * **Edad:** Su OR es superior a 1, reiterando el aumento progresivo del riesgo por cada año cumplido.

* **Factores Protectores (Odds Ratio < 1):**
  * **Miembro Activo (OR = 0.3411):** Estar activo reduce sustancialmente la posibilidad de abandono. Los clientes activos tienen aproximadamente un 65.9% menos posibilidades relativas de irse comparados con los inactivos.
  * **Sexo Masculino (OR < 1):** Ser hombre actúa como un factor protector frente al perfil femenino de referencia.
  * **Número de Productos (OR = 0.9035):** Mantener un mayor número de productos disminuye en casi un 10% las posibilidades de abandonar por cada producto adicional.

In [7]:
from sklearn.metrics import confusion_matrix, classification_report

# Partición 70 / 30 con semilla fija numpy
rng = np.random.default_rng(8)
shuffled_indices = rng.permutation(len(df))
train_size = int(0.7 * len(df))

train_idx = shuffled_indices[:train_size]
test_idx = shuffled_indices[train_size:]

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# Re-estimar modelo en Train
m_train = smf.logit(formula, data=train_df).fit()

# Predicción de probabilidades en Test
prob_test = m_train.predict(test_df)

# Evaluación con umbrales 0.5 y 0.4
for umbral in [0.5, 0.4]:
    pred_clase = (prob_test >= umbral).astype(int)
    cm = confusion_matrix(test_df["abandono"], pred_clase)
    print(f"\n================ Matriz de Confusión (Umbral = {umbral}) ================")
    print(cm)
    print("\nReporte de Clasificación:")
    print(classification_report(test_df["abandono"], pred_clase, target_names=["Permanece (0)", "Abandona (1)"]))

Optimization terminated successfully.
         Current function value: 0.431640
         Iterations 6

================ Matriz de Confusión (Umbral = 0.5) ================
[[2307   87]
 [ 466  140]]

Reporte de Clasificación:
               precision    recall  f1-score   support

Permanece (0)       0.83      0.96      0.89      2394
 Abandona (1)       0.62      0.23      0.34       606

     accuracy                           0.82      3000
    macro avg       0.72      0.60      0.61      3000
 weighted avg       0.79      0.82      0.78      3000


================ Matriz de Confusión (Umbral = 0.4) ================
[[2211  183]
 [ 404  202]]

Reporte de Clasificación:
               precision    recall  f1-score   support

Permanece (0)       0.85      0.92      0.88      2394
 Abandona (1)       0.52      0.33      0.41       606

     accuracy                           0.80      3000
    macro avg       0.69      0.63      0.65      3000
 weighted avg       0.78      0.80      

## Sección 6: Evaluación del Modelo en Prueba y Comparación de Umbrales (0.5 vs 0.4)

Para evaluar el desempeño del modelo Logit en datos no vistos, utilicé la muestra de prueba de 3,000 observaciones y probé dos umbrales de decisión distintos: el estándar de 0.5 y un umbral más sensible de 0.4.

### 1. Evaluación con Umbral Estándar (0.5)
* **Matriz de Confusión:**
  * Verdaderos Negativos (Permanecen correctamente clasificados): 2,307
  * Falsos Positivos (Se predijo que se iban, pero se quedaron): 87
  * Falsos Negativos (Se predijo que se quedaban, pero se fueron): 466
  * Verdaderos Positivos (Se predijo correctamente que se iban): 140
* **Métricas Principales:**
  * **Exactitud (Accuracy):** 82%. A primera vista parece alta, pero se debe a que la mayoría de los clientes no abandonan.
  * **Sensibilidad (Recall clase 1):** Solo alcanza el 23%. Con un umbral de 0.5, el modelo apenas detecta a 140 de los 606 clientes que realmente se fugaron, dejando escapar al 77% de los clientes en riesgo.

---

### 2. Evaluación con Umbral Ajustado (0.4)
* **Matriz de Confusión:**
  * Verdaderos Negativos: 2,211
  * Falsos Positivos: 183
  * Falsos Negativos: 404
  * Verdaderos Positivos: 202
* **Métricas Principales:**
  * **Exactitud (Accuracy):** Baja ligeramente a 80%.
  * **Sensibilidad (Recall clase 1):** Aumenta al 33.3% (detecta 202 de 606 casos reales de abandono).

---

### 3. Conclusión sobre el Cambio de Umbral
Reducir el umbral de 0.5 a 0.4 resulta ser una mejor estrategia comercial para el área de retención del banco. Aunque se sacrifican 2 puntos de exactitud general (pasa de 82% a 80%) y aumentan los falsos positivos (de 87 a 183), se logra identificar a 62 clientes adicionales en riesgo de abandono (pasa de 140 a 202 verdaderos positivos). En un contexto de retención bancaria, el costo de perder a un cliente no detectado suele ser mucho mayor que el costo de contactar preventivamente a un cliente que no planeaba irse.